In [1]:
import pandas as pd
import numpy as np
import ast
import re, string
import json
import csv
import bs4
from bs4 import BeautifulSoup
import requests
import time
import random
from tqdm import tqdm
import spacy
import gensim
import nltk
from nltk.probability import FreqDist
from nltk.corpus import stopwords
import wordcloud 
# ! python -m spacy download en_core_web_sm
nlp = spacy.load("en_core_web_sm")

# data viz
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap as LSC
import seaborn as sns
import sklearn.manifold #For T-SNE
import sklearn.decomposition #For PCA

In [ ]:
nonenglish_comms = r"""[^a-zA-Z\d\s\[\]\-\#\.\?\,\&\<\>\!\@\$\%\^\*\+\=\:\;\\/\%\'\"]"""

# create custom color map for graphs
colors = ['#800000', '#D9D9D9', '#A6A6A6', '#737373']
cmap = LSC.from_list("custom_colormap", colors)

In [ ]:
alj_yt = pd.read_csv("D:\\hw\\macs-30122\\final-project-chattbd\\output_YT_AlJazeera.csv")
cnn_yt = pd.read_csv("D:\\hw\\macs-30122\\final-project-chattbd\\output_YT_CNN_approx252videos.csv")
fox_yt = pd.read_csv("D:\\hw\\macs-30122\\final-project-chattbd\\output_YT_FoxNews.csv")

In [ ]:
print("ALJ news:", alj_yt.shape)
print("CNN news:", cnn_yt.shape)
print("FOX news:", fox_yt.shape)

### Helper funcs from other classes

In [ ]:
# function from HW2 and HW4 of Content Analysis

def word_tokenize(word_list):
    """
    Take a list of words and tokenize the text
    Input: list of strs
    Returns a list of tokenized strs 
    """
    tokenized = []
    # pass word list through language model.
    doc = nlp(word_list)
    for token in doc:
        if not token.is_punct and len(token.text.strip()) > 0:
            tokenized.append(token.text)
    return tokenized


def normalizeTokens(word_list, extra_stop=[]):
    """
    Takes a list of words and normalizes the tokens
    Inputs:
        word_list: list of strs for words needed to be tokenized
        extra_stop: list of strs for extra stop words
    Returns list of tokenized strs
    """
    #We can use a generator here as we just need to iterate over it
    normalized = []
    if type(word_list) == list and len(word_list) == 1:
        word_list = word_list[0]

    if type(word_list) == list:
        word_list = ' '.join([str(elem) for elem in word_list])

    doc = nlp(word_list.lower())

    # add the property of stop word to words considered as stop words
    if len(extra_stop) > 0:
        for stopword in extra_stop:
            lexeme = nlp.vocab[stopword]
            lexeme.is_stop = True

    for w in doc:
        # if it's not a stop word or punctuation mark, add it to our article
        if w.text != '\n' and not w.is_stop and not w.is_punct \
            and not w.like_num and len(w.text.strip()) > 0:
            # we add the lematized version of the word
            normalized.append(str(w.lemma_))

    return normalized


def sent_tokenize(word_list, model=nlp):
    doc = model(word_list)
    sentences = [sent.text.strip() for sent in doc.sents]
    return sentences


# EDITED
# functions from TextClassification.ipynb MACS 30100

def decontracted(phrase):
    """
    Expand the contracted phrase into normal words
    """
    # specific
    phrase = re.sub(r"won't", "will not", phrase)
    phrase = re.sub(r"can\'t", "can not", phrase)
    # general
    phrase = re.sub(r"n\'t", " not", phrase)
    phrase = re.sub(r"\'re", " are", phrase)
    phrase = re.sub(r"\'s", " is", phrase) # prime 
    phrase = re.sub(r"\'d", " would", phrase)
    phrase = re.sub(r"\'ll", " will", phrase)
    phrase = re.sub(r"\'t", " not", phrase)
    phrase = re.sub(r"\'ve", " have", phrase)
    phrase = re.sub(r"\'m", " am", phrase)
    
    return phrase


def tag_sents_pos(sentences):
    """
    function which replicates NLTK pos tagging on sentences.
    """
    new_sents = []
    for sentence in sentences:
        new_sent = ' '.join(sentence)
        new_sents.append(new_sent)
    final_string = ' '.join(new_sents)
    doc = nlp(final_string)

    pos_sents = []
    for sent in doc.sents:
        pos_sent = []
        for token in sent:
            pos_sent.append((token.text, token.tag_))
        pos_sents.append(pos_sent)

    return pos_sents

### My helpers

In [ ]:
def clean_transcript(transcript):
    """
    Takes a string that contains a transcript and removes extraneous noise,
    Including timestamps, CC information, etc, some other weird.
    Inputs:
        transcript: str
    Returns a cleaned transcript with text in lower case
    """

    # if auto generated in Arabic it would be much harder to perform our
    # task, so for simplicity we will remove such transcripts
    if re.search(r"Arabic \(auto-generated\)$", transcript):
        return []
    
    # speaker cites
    speaker = r">>\s*([\w]+):|(?<=\d{2}\s)(\w+)(?=\:)"
    # find the noise in the text, formatting things mostly
    noise = r"(>{1,2}|♪|\b(\d{1,2}\:)*\d{1,2}\:\d{2}\b)"
    # find the transcript formatting things that appear at the end
    cc = r"(?:English \(auto-generated\)|English|English \- cc1)\s*$"
    
    # remove artifacts of being closed captioning: speaker attr, timestamps, etc
    transcript = re.sub(speaker, "", transcript, flags=re.IGNORECASE)
    transcript = re.sub(noise, " ", transcript, flags=re.IGNORECASE)
    transcript = re.sub(cc, "", transcript, flags=re.IGNORECASE)
    # remove contractions
    transcript = decontracted(transcript)
    # remove the extra spaces, strip whitespace and lower case
    return re.sub(r"\s+", " ", transcript).strip().lower()


def clean_comm(comments, removers="", not_jank=True):
    """
    Cleans up the comment section, removing non-English symbols and URLs,
    among other things. For jankier comment sections removes more formatting also
    Input: 
        comments: a string that appears formatted as a list of strs
        removers: symbols to remove
        not_jank: bool to track whether the comments need more cleaning
    Returns a str combininb all cleaned comments
    """
    # remove contractions
    comm_str = decontracted(" ".join(ast.literal_eval(comments)))
    # remove html tags and URLs
    comm_str = BeautifulSoup(comm_str, 'lxml').get_text().strip() 
    comm_str = re.sub(r'<.*?>|https*://\S+|www\.\S+', '', comm_str)
    # remove problematic characters
    comm_str = re.sub(removers, "", comm_str)

    # should only be jank for CNN
    if not_jank:
        # return the comment as a 
        return comm_str.strip().lower()
    else:
        comment_clean = r"(?<=ago)\s*(?:\n|\(edited\)\n)(.*?)(?:Read more)*(?=\n(?:\d+\n)?Reply)"
        # I was having issues capturing all comments until I re.DOTALL
        # https://chat.openai.com/share/ed2187c5-dbcf-4a34-ac18-41e54890c812
        # consulted ChatGPT to help resolve this issue (using DOTALL)
        return " ".join(re.findall(comment_clean, comm_str, 
                                   flags=re.DOTALL)).strip().lower()


def clean_title(title, removers=""):
    """
    Cleans up YT titles
    Input: 
        title: a string corresponding to a video title
        removers: symbols to remove
    Returns cleaned title
    """
    title = decontracted(title)
    title = BeautifulSoup(title, 'lxml').get_text().strip() 
    title = re.sub(r'<.*?>|https*://\S+|www\.\S+', '', title)
    # remove problematic characters
    title = re.sub(removers, "", title)
    return title.strip().lower()


### Actual Data Cleaning

#### Youtube

##### RUN ONCE

In [ ]:
def clean_yt(dfs,file_saves=["alj_yt.pkl", "cnn_yt.pkl", "fox_yt.pkl"]):
    """
    Take a list or iterable of pandas.DataFrames and edit them in place
    Essentially this function performs multiple tasks, it reformats columns,
    and cleans the data so it is more ready for textual analyses in the future
    Inputs:
        dfs: iterable of pandas.DataFrames
        file_saves: list of strs for filenames
    Returns None, edits DFs in place
    """
    # for progress bar
    tqdm.pandas()

    for i, df in enumerate(dfs):
        # change to datetime
        df["date"] = pd.to_datetime(df["date"])
        
        # replace all data that is uninformative with actually NaN values
        # https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.replace.html
        df.replace({'transcript': {'NA': np.nan,
                                "NaN": np.nan,
                                'Unable to obtain transcript': np.nan},
                    'comments': {'NA': np.nan, 
                                "NaN": np.nan,
                                "[]": np.nan}
                    }, inplace=True)
        
        # remove uninformative rows (no info in transcript or comments)
        df.drop(df[(df['transcript'].isna()) & 
            (df['comments'].isna())].index,inplace=True)

        # clean comments up
        nonenglish_comms = r"""[^a-zA-Z\d\s\[\]\-\#\.\?\,\&\<\>\!\@\$\%\^\*\+\=\:\;\\/\%\'\"]"""
        
        #the bool in clean_comm determines if more cleaning is necessary
        # based off whether the comment str starts with an "@"
        jank_start = re.compile(r"^((?:\[[\'\"])@)")
        df.comments = df.comments.apply(lambda x: 
                                        clean_comm(x, nonenglish_comms, 
                                        bool(not jank_start.match(x))) 
                                        if pd.notnull(x) else x)
        
        # clean transcript up, only run if not null
        df.transcript = df.transcript.apply(lambda x: clean_transcript(x)
                                            if pd.notnull(x) else x)
        # clean title
        df.title = df.title.apply(lambda x: clean_title(x, nonenglish_comms))
        # find useful text: a mix of title and transcript
        df["text"] = df.title + ". " + df.transcript.apply(lambda x: x if pd.notna(x) else "")
        # drop channel column (contains duplicate info), drop title, transcript
        df.drop(['channel',"title", "transcript"], axis=1, inplace=True)
        
        # tokenize and normalize comments
        df["toke_comms"] = df['comments'].progress_apply(lambda x: \
                           word_tokenize(x) if pd.notna(x) else x)
        df['word_counts_comms'] = df['toke_comms'].progress_apply(lambda x: \
                                  len(x) if isinstance(x, list) else x)
        df['norm_comms'] = df['toke_comms'].progress_apply(lambda x: \
                           normalizeTokens(x) if isinstance(x, list) else x)
        df['toke_comm_sents'] = df['comments'].progress_apply(lambda x: \
                                [word_tokenize(s) for s in sent_tokenize(x)] \
                                if pd.notna(x) else x)
        df['norm_comm_sents'] = df['toke_comm_sents'].progress_apply(lambda x: \
                                [normalizeTokens(s) for s in x] if isinstance(x, list) else x)
        
        # tokenize and normalize the texts
        df["toke_text"] = df['text'].progress_apply(lambda x: word_tokenize(x))
        df['word_counts'] = df['toke_text'].progress_apply(lambda x: len(x))
        df['norm_tokens'] = df['toke_text'].progress_apply(lambda x: \
                            normalizeTokens(x))
        df['toke_sents'] = df['text'].progress_apply(lambda x: \
                           [word_tokenize(s) for s in sent_tokenize(x)])
        df['norm_sents'] = df['toke_sents'].progress_apply(lambda x: \
                           [normalizeTokens(s) for s in x])

        # reset the index
        df.reset_index(drop=True, inplace=True)
        df.to_pickle(file_saves[i])
    return

In [ ]:
clean_yt([alj_yt, cnn_yt, fox_yt])

##### Load in Data (so not have to run above constantly)

In [ ]:
# alj_yt = pd.read_pickle("D:\\hw\\macs-30122\\final-project-chattbd\\alj_yt.pkl")
# cnn_yt = pd.read_pickle("D:\\hw\\macs-30122\\final-project-chattbd\\cnn_yt.pkl")
# fox_yt = pd.read_pickle("D:\\hw\\macs-30122\\final-project-chattbd\\fox_yt.pkl")

full_yt = pd.read_pickle("D:\\hw\\macs-30122\\final-project-chattbd\\full_yt.pkl")

##### Lengths

In [ ]:
# FOR COMMENTS SM
print("ALJ comms:", alj_yt.comments.notna().sum())
print("CNN comms:", cnn_yt.comments.notna().sum()) 
print("FOX comms:", fox_yt.comments.notna().sum())

# FOR NEWS
print("ALJ news:", alj_yt.text.notna().sum())
print("CNN news:", cnn_yt.text.notna().sum())
print("FOX news:", fox_yt.text.notna().sum())

#### Reddit


##### RUN ONCE

In [ ]:
# read the pkl file
ip_df = pd.read_pickle("israel_palestine_df.pkl")

In [ ]:
ip_df = pd.read_pickle("ip_df_new.pkl")

In [ ]:
# drop rows with no useful info
ip_df.dropna(subset=['post_text'], inplace=True)

In [ ]:
# clean up the post a little
ip_df.post_text = ip_df.post_text.apply(lambda x: clean_title(x, removers=nonenglish_comms))

In [ ]:
# normalize tokens and sents
tqdm.pandas()
ip_df["toke_text"] = ip_df['post_text'].progress_apply(lambda x: word_tokenize(x))
ip_df['word_counts'] = ip_df['toke_text'].progress_apply(lambda x: len(x))
ip_df['norm_tokens'] = ip_df['toke_text'].progress_apply(lambda x: normalizeTokens(x))
ip_df['toke_sents'] = ip_df['post_text'].progress_apply(lambda x: [word_tokenize(s) for s in sent_tokenize(x)])
ip_df['norm_sents'] = ip_df['toke_sents'].progress_apply(lambda x: [normalizeTokens(s) for s in x])

In [ ]:
# save files
ip_df.to_pickle("ip_df_new.pkl")

In [ ]:
start_date = '2023-10-07'
end_date = '2023-12-31'
ip_df[(ip_df["post_date"] >= start_date) & (ip_df["post_date"] <= end_date)]

ip_df.to_pickle("ip_df_timed.pkl")

##### Reopen file

In [ ]:
ip_df = pd.read_pickle("D:\\hw\\macs-30122\\final-project-chattbd\\ip_df_timed.pkl")

### News

#### Run once

In [ ]:
fox_news = pd.read_pickle("D:\\hw\\macs-30122\\final-project-chattbd\\fox_articles.pkl")
# drop these useless columns
fox_news.drop(columns=['Unnamed: 0'], inplace=True)
# drop any possible duplicate rows based off of article url
fox_news.drop_duplicates(subset=['article_url'], inplace=True)
# drop rows with nas in all_text
fox_news.dropna(subset=["all_text"], inplace=True)
# reset indices
fox_news.reset_index(drop=True, inplace=True)
# add title to all_text
fox_news["text"] = fox_news.article_title + ". " + fox_news.all_text
# clean up the text a little
fox_news["text"] = fox_news.text.apply(lambda x: clean_title(x, nonenglish_comms))

In [ ]:
# tokenization and normalization
tqdm.pandas()
fox_news["toke_text"] = fox_news['text'].progress_apply(lambda x: word_tokenize(x))
fox_news['word_counts'] = fox_news['toke_text'].progress_apply(lambda x: len(x))
fox_news['norm_tokens'] = fox_news['toke_text'].progress_apply(lambda x: \
                    normalizeTokens(x))
fox_news['toke_sents'] = fox_news['text'].progress_apply(lambda x: \
                    [word_tokenize(s) for s in sent_tokenize(x)])
fox_news['norm_sents'] = fox_news['toke_sents'].progress_apply(lambda x: \
                    [normalizeTokens(s) for s in x])

fox_news.reset_index(drop=True, inplace=True)

In [ ]:
fox_news.to_pickle('fox_news.pkl')

#### open file

In [ ]:
fox_news = pd.read_pickle("fox_news.pkl")
all_news = pd.read_pickle("all_news.pkl")

#### Forgot to add POS tagger to all data (don't run again)

In [ ]:
# combine all yt channels together
full_yt = pd.concat([alj_yt, cnn_yt, fox_yt])
full_yt.reset_index(drop=True, inplace=True)

In [ ]:
tqdm.pandas()
full_yt['POS_sents'] = full_yt.toke_sents.progress_apply(lambda x: \
    tag_sents_pos(x) if isinstance(x, list) else x)
full_yt['POS_comm_sents'] = full_yt.toke_comm_sents.progress_apply(lambda x: \
    tag_sents_pos(x) if isinstance(x, list) else x)
fox_news["POS_sents"] = fox_news.toke_sents.progress_apply(lambda x: tag_sents_pos(x))
ip_df["POS_sents"] = ip_df.toke_sents.progress_apply(lambda x: tag_sents_pos(x))

In [ ]:
full_yt.to_pickle("full_yt.pkl")
fox_news.to_pickle('fox_news.pkl')
ip_df.to_pickle("ip_df_new.pkl")

### Word2Vec


#### Agg SM and News (Run once)

In [ ]:
full_yt = pd.read_pickle("D:\\hw\\macs-30122\\final-project-chattbd\\full_yt.pkl")

In [ ]:
# turning all social media data into one dataframe
all_sm = pd.concat([full_yt.drop(["channel_key", "date", "text", "toke_text", 
                                   "word_counts", "norm_tokens", 
                                   "toke_sents", "norm_sents", "POS_sents"], 
                                   axis=1).rename(columns={"comments":"text",
                                                   "toke_comms":"toke_text",
                                                   "word_counts_comms":"word_counts",
                                                   "norm_comms":"norm_tokens",
                                                   'toke_comm_sents':"toke_sents",
                                                   "norm_comm_sents":"norm_sents",
                                                   "POS_comm_sents":"POS_sents"}
                                                   ).dropna(),
                    ip_df.rename(columns={'post_text': 
                                          'text'}).drop(["user", "user_flair", 
                                                         "title", "post_date", 
                                                         "post_flair", 
                                                         "score", "n_comments",
                                                         "link", "is_comment", 
                                                         "sub"], axis=1)]
                                                         ).reset_index(drop=True)

all_sm.to_pickle("all_sm.pkl")

In [ ]:
# turning all social media data into one dataframe
all_news = pd.concat([full_yt.drop(['channel_key', 'date', "comments","toke_comms", 
                                    "word_counts_comms", "norm_comms", "toke_comm_sents", 
                                    "norm_comm_sents", "POS_comm_sents"], axis=1).dropna(), 
                      fox_news.drop(columns=["article_title", "all_text", 
                                             "article_url", "article_date"],
                                             axis=1)]).reset_index(drop=True)

all_news.to_pickle("all_news.pkl")

#### Word2Vec model

#### Make word2vec models (Run Once)

In [ ]:
all_sm = pd.read_pickle("D:\\hw\\macs-30122\\final-project-chattbd\\all_sm.pkl")
all_news = pd.read_pickle("D:\\hw\\macs-30122\\final-project-chattbd\\all_news.pkl")    

In [ ]:
social_media_model = gensim.models.word2vec.Word2Vec(all_sm.norm_sents.sum())
social_media_model.save("sm_word2vec.model")

In [ ]:
news_model = gensim.models.word2vec.Word2Vec(all_news.norm_sents.sum())
news_model.save("news_word2vec.model")

#### Load Data and Models

In [ ]:
all_sm = pd.read_pickle("D:\\hw\\macs-30122\\final-project-chattbd\\all_sm.pkl")
fox_news = pd.read_pickle("D:\\hw\\macs-30122\\final-project-chattbd\\fox_news.pkl")
social_media_model = gensim.models.Word2Vec.load("sm_word2vec.model")
news_model = gensim.models.Word2Vec.load("news_word2vec.model")

In [ ]:
# for idea how to collapse the list of lists into one list
# https://stackoverflow.com/questions/30885005/pandas-series-of-lists-to-one-series
# create a series of all POS tags for each word in the social media data
all_sm_POS = pd.Series(all_sm.POS_sents.explode().explode().dropna()).reset_index(drop=True)

news_POS = pd.Series(fox_news.POS_sents.explode().explode().dropna()).reset_index(drop=True)


In [ ]:
# determine common English stopwords
stop_words = stopwords.words('english')
# remove stop words
sm_POS_no_stops = all_sm_POS[~all_sm_POS.apply(lambda x: x[0] in stop_words)]

### DATA VIZ

In [ ]:
# find the top 100 most common tokens (normalized)
word_freq = FreqDist(all_sm.norm_tokens.explode())
top_100 = pd.DataFrame(word_freq.most_common(100), columns=["word", "freq"])

# # plot top 100 words
# fig = plt.figure()
# ax = fig.add_subplot(111)
# plt.plot(range(len(top_100)), top_100.freq)
# plt.xticks()
# plt.show()

# # Zipf's law 
# fig = plt.figure()
# ax = fig.add_subplot(111)
# plt.plot(range(len(top_100)), top_100.freq)
# ax.set_yscale('log')
# ax.set_xscale('log')
# plt.show()

In [ ]:
all_sm_tokens = list(all_sm.toke_text.explode().dropna())
all_sm_Text = nltk.Text(all_sm_tokens)
all_sm_index = nltk.text.ConcordanceIndex(all_sm_Text)

In [ ]:
all_sm_index.print_concordance('palestine', width=60, lines=25)

In [ ]:
all_sm_index.print_concordance('israel', width=60, lines=25)

In [ ]:
# put all normalized words into one series
all_sm_norm = all_sm.norm_tokens.explode().dropna().reset_index(drop=True)
# remove stop words
all_sm_norm_non = all_sm_norm[~all_sm_norm.apply(lambda x: x in stop_words)]

In [ ]:
# load news
fox_news = pd.read_pickle("D:\\hw\\macs-30122\\final-project-chattbd\\fox_news.pkl")

In [ ]:
news_norm = fox_news.norm_tokens.explode().dropna().reset_index(drop=True)
# remove stop words
news_norm_non = news_norm[~news_norm.apply(lambda x: x in stop_words)]

##### Find the Number of Words used in Social Media and News Sources

In [ ]:
print("SM words:", all_sm.toke_text.explode().dropna().reset_index(drop=True).size)
print("SM unique words:", len(all_sm.toke_text.explode().dropna().reset_index(drop=True).unique()))
print("SM Vocab Density:", round(len(all_sm.toke_text.explode().dropna().reset_index(drop=True).unique()) / 
                                 all_sm.toke_text.explode().dropna().reset_index(drop=True).size, 3))
print("News words:", fox_news.toke_text.explode().dropna().reset_index(drop=True).size)
print("News unique words:", len(fox_news.toke_text.explode().dropna().reset_index(drop=True).unique()))
print("News  Vocab Density:", round(len(fox_news.toke_text.explode().dropna().reset_index(drop=True).unique()) / 
                                    fox_news.toke_text.explode().dropna().reset_index(drop=True).size, 3))


In [ ]:
print("SM normalized words:", all_sm_norm_non.size)
print("SM normalized unique words:", len(all_sm_norm_non.unique()))
print("SM normalized Vocab Density:", round(len(all_sm_norm_non.unique()) / all_sm_norm_non.size,3))

print("News normalized words:", news_norm_non.size)
print("News normalized unique words:", len(news_norm_non.unique()))
print("News normalized Vocab Density:", round(len(news_norm_non.unique()) / news_norm_non.size,3))

In [ ]:
import wordcloud
# Create a WordCloud object with specified parameters
wc = wordcloud.WordCloud(background_color="white", max_words=100, width=1000,
                         colormap=cmap,height=1000, mode='RGBA', scale=.5
                         ).generate(' '.join(all_sm_norm_non))

plt.figure(figsize=(10, 6))
plt.imshow(wc)
plt.axis("off")
plt.show()

In [ ]:
# WORD CLOUD for News

# Create a WordCloud object with specified parameters
wc = wordcloud.WordCloud(background_color="white", max_words=100, width=1000,
                         colormap=cmap,height=1000, mode='RGBA', scale=.5
                         ).generate(' '.join(news_norm_non))

plt.figure(figsize=(10, 6))
plt.imshow(wc)
plt.axis("off")
plt.show()

In [ ]:
# cond Freq Dist for all POS tags
all_sm_cfdist_POStoWord = nltk.ConditionalFreqDist((p, w) for w, p in sm_POS_no_stops)
# Frequency distribution
freq_dist_nnps = all_sm_cfdist_POStoWord["NNPS"]

In [ ]:
freq_dist_nnp = all_sm_cfdist_POStoWord["NNP"]
# word cloud for proper nouns
wc = wordcloud.WordCloud(width=1000, height=1000, max_words=50,
                                colormap=cmap, background_color='white', 
                                mode='RGBA', scale=.5
                                ).generate_from_frequencies(freq_dist_nnp)



# Display the word cloud
plt.figure(figsize=(10, 6))
plt.imshow(wc)
plt.axis('off')
plt.show()

In [ ]:
# cond Freq Dist for all POS tags
news_cfdist_POStoWord = nltk.ConditionalFreqDist((p, w) for w, p in sm_POS_no_stops)

In [ ]:
# Frequency distribution
freq_dist_nnp = all_sm_cfdist_POStoWord["NNP"]
# word cloud for proper nouns
wc = wordcloud.WordCloud(width=1000, height=1000, max_words=50,
                                colormap=cmap, background_color='white', 
                                mode='RGBA', scale=.5
                                ).generate_from_frequencies(freq_dist_nnp)



# Display the word cloud
plt.figure(figsize=(10, 6))
plt.imshow(wc)
plt.axis('off')
plt.show()

In [ ]:
import wordcloud

freq_dist_vb = all_sm_cfdist_POStoWord["VB"]
# word cloud for proper nouns
wc = wordcloud.WordCloud(width=1000, height=1000, max_words=100,
                         colormap=cmap,background_color='white', mode='RGBA', 
                         scale=.5).generate_from_frequencies(freq_dist_vb)

# Display the word cloud
plt.figure(figsize=(10, 6))
plt.imshow(wc)
plt.axis('off')
plt.show()

#### word to vec viz

In [ ]:
targetWords = social_media_model.wv.index_to_key[:75]

# create a submatrix
wordsSubMatrix = []
for word in targetWords:
    wordsSubMatrix.append(social_media_model.wv[word])
wordsSubMatrix = np.array(wordsSubMatrix)
wordsSubMatrix


pcaWords = sklearn.decomposition.PCA(n_components = 75).fit(wordsSubMatrix)
reducedPCA_data = pcaWords.transform(wordsSubMatrix)
#T-SNE is theoretically better, but you should experiment
tsneWords = sklearn.manifold.TSNE(n_components = 2).fit_transform(reducedPCA_data)

In [ ]:
pcaWords = sklearn.decomposition.PCA(n_components = 75,random_state=42).fit_transform(wordsSubMatrix)

fig = plt.figure(figsize = (10,6))
ax = fig.add_subplot(111)

plt.scatter(pcaWords[:, 0], pcaWords[:, 1], alpha=0.5, color="#737373")
for i, word in enumerate(targetWords):
    plt.annotate(word, (pcaWords[:, 0][i], pcaWords[:, 1][i]), 
                 size=20 * (100 - i) / 100, color="#800000", alpha=0.75)

plt.xticks(())
plt.yticks(())

plt.axis('off')
plt.show()

In [ ]:
tsneWords = sklearn.manifold.TSNE(n_components = 2, random_state=69).fit_transform(wordsSubMatrix)

# lets plot the top 50 words
fig = plt.figure(figsize = (10,6))
ax = fig.add_subplot(111)

plt.scatter(tsneWords[:, 0], tsneWords[:, 1], alpha=0.5, color="#737373")
for i, word in enumerate(targetWords):
    plt.annotate(word, (tsneWords[:, 0][i],tsneWords[:, 1][i]), 
                 size =  20 * (100 - i) / 100, color="#800000", alpha=0.75)

plt.xticks(())
plt.yticks(())
plt.axis('off')
plt.show()